In [1]:
import sys, os
ROOT = "/Users/fserracrespi/Desktop/PD_PROJECT_UOFL" 
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)
from utils.Decode import decoder_DF
from utils.Prog_df import check_progression
from utils.dataframe_cols import cols_asignacion2
from utils.Prog_df import check_progression_multi
from utils.Prog_df import check_visits
import json
from collections import defaultdict
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from scipy.stats import kurtosis, skew
import re
import math

In [2]:
path2='/Users/fserracrespi/Desktop/PD_Project_UofL/DATA_COLLECTION/12_09_2025/Study_Docs/Data___Databases/DATA/Data_Dictionary_-_Harmonized_12Sep2025.csv'
path3='/Users/fserracrespi/Desktop/PD_Project_UofL/DATA_COLLECTION/12_09_2025/Study_Docs/Data___Databases/DATA/Code_List_-_Harmonized_12Sep2025.csv'
code_cols=pd.read_csv(path2, dtype=str)
code_rows=pd.read_csv(path3, dtype=str)

# Initial Data Load

## Presence Matrix PD


In [3]:
PATNOs = pd.read_csv('/Users/fserracrespi/Desktop/PD_Project_UofL/PD_DATA/PD_CSV_CLEAN/Relevant_presnece_PATNO', dtype=str)['PATNO'].tolist()
print(f'Total PATNOs to process: {len(PATNOs)}')

Total PATNOs to process: 1056


## Subject Data 

In [4]:
# Initial Data Load
subject_df = pd.read_csv('/Users/fserracrespi/Desktop/PD_Project_UofL/DATA_COLLECTION/12_09_2025/Subject Characteristics/Patient_Status/DATA/Participant_Status_12Sep2025.csv', dtype=str)
subject_df=decoder_DF(subject_df, code_rows, code_cols, module='PATIENT_STATUS')
subject_df = subject_df[subject_df['PATNO'].isin(PATNOs)]
print(f'Subjects after filtering: {subject_df.shape[0]}')
subject_df

Subjects after filtering: 1056


/Users/fserracrespi/Desktop/PD_PROJECT_UOFL/utils/Decode.py:81: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df[col].replace('NaN', np.nan, inplace=True)


,PATNO,Enrollment Cohort,Decoded Value for COHORT,Enrollment Date,Enrollment Status,Date Enrollment Status Occurred,PPMI Clinical Amendment Participant was Screened under,Age at Enrollment,PPMI Clinical Inclusion/Criteria,Participant in PPMI Clinical Early Imaging Sub-Study,...,Pink1 Mutation at Enrollment,Parkin Mutation at Enrollment,Sporadic PD at Enrollment,Normosmic PD at Enrollment,Other Genetic Variant at Enrollment,Hyposmia / Generalized Risk at Enrollment,RBD at Enrollment,LRRK2 Mutation at Enrollment,SNCA Mutation at Enrollment,GBA Mutation at Enrollment
1,3001,Parkinson's Disease,Parkinson's Disease,03/2011,Enrolled,09/2021,NaN,65.1,NaN,No,...,No,No,Yes,NaN,NaN,No,No,No,No,No
3,3003,Parkinson's Disease,Parkinson's Disease,04/2011,Enrolled,01/2022,NaN,56.7,NaN,No,...,No,No,Yes,NaN,NaN,No,No,No,No,No
10,3010,Parkinson's Disease,Parkinson's Disease,06/2011,Enrolled,05/2021,NaN,47.0,NaN,No,...,No,No,Yes,NaN,NaN,No,No,No,No,No
18,3018,Parkinson's Disease,Parkinson's Disease,04/2012,Enrolled,07/2021,NaN,60.6,NaN,No,...,No,No,Yes,NaN,NaN,No,No,No,No,No
21,3021,Parkinson's Disease,Parkinson's Disease,05/2012,Enrolled,06/2021,NaN,64.1,NaN,No,...,No,No,Yes,NaN,NaN,No,No,No,No,No
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7469,446733,Parkinson's Disease,Parkinson's Disease,05/2025,Baseline Withdraw,08/2025,4,71.5,Parkinson's Disease (Normosmic) Inclusion/Excl...,No,...,No,No,No,Yes,No,No,No,No,No,No
7480,452611,Parkinson's Disease,Parkinson's Disease,NaN,Baseline,09/2025,3,NaN,Parkinson's Disease (SNCA or Parkin) Inclusion...,No,...,No,No,No,No,Yes,No,No,No,No,No
7489,466157,Parkinson's Disease,Parkinson's Disease,05/2025,Enrolled,05/2025,4,28.3,Parkinson's Disease (SNCA or Parkin) Inclusion...,No,...,No,Yes,No,No,NaN,No,No,No,No,No
7497,476236,Parkinson's Disease,Parkinson's Disease,08/2025,Enrolled,08/2025,4,59.5,Parkinson's Disease (Normosmic) Inclusion/Excl...,No,...,No,No,No,Yes,No,No,No,No,No,No


In [5]:
list_relevant_cols=['PATNO', 'Sporadic PD at Enrollment','RBD at Enrollment',
                    'Pink1 Mutation at Enrollment', 'Parkin Mutation at Enrollment','LRRK2 Mutation at Enrollment', 'SNCA Mutation at Enrollment',
                    'GBA Mutation at Enrollment']

relevant_subject_df=subject_df[list_relevant_cols]
print(f'Relevant subject dataframe shape: {relevant_subject_df.shape}')
print(f'NaN values in relevant subject dataframe:\n{relevant_subject_df.isna().sum()}')


Relevant subject dataframe shape: (1056, 8)
NaN values in relevant subject dataframe:
PATNO                            0
Sporadic PD at Enrollment        0
RBD at Enrollment                0
Pink1 Mutation at Enrollment     0
Parkin Mutation at Enrollment    0
LRRK2 Mutation at Enrollment     0
SNCA Mutation at Enrollment      0
GBA Mutation at Enrollment       0
dtype: int64


In [6]:
for col in relevant_subject_df.columns[1:]:
    relevant_subject_df[col] = relevant_subject_df[col].map({'Yes': 1, 'No': 0})# Convert 'Yes'/'No' to 1/0

/var/folders/vr/_v9wq92941bflv75jsh8kq800000gn/T/ipykernel_47989/3051402693.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  relevant_subject_df[col] = relevant_subject_df[col].map({'Yes': 1, 'No': 0})# Convert 'Yes'/'No' to 1/0
/var/folders/vr/_v9wq92941bflv75jsh8kq800000gn/T/ipykernel_47989/3051402693.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  relevant_subject_df[col] = relevant_subject_df[col].map({'Yes': 1, 'No': 0})# Convert 'Yes'/'No' to 1/0
/var/folders/vr/_v9wq92941bflv75jsh8kq800000gn/T

## Demographic

In [7]:
demo_df = pd.read_csv('/Users/fserracrespi/Desktop/PD_Project_UofL/DATA_COLLECTION/12_09_2025/Subject Characteristics/Subject_Demographics/DATA/Demographics_12Sep2025.csv', dtype=str)
demo_df=decoder_DF(demo_df, code_rows, code_cols, module='SCREEN')
demo_df = demo_df[demo_df['PATNO'].isin(PATNOs)]
demo_df.columns

/Users/fserracrespi/Desktop/PD_PROJECT_UOFL/utils/Decode.py:81: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df[col].replace('NaN', np.nan, inplace=True)


Index(['Record ID', 'PATNO', 'Visit ID', 'Page Name', 'Site Aware Date',
       'Identify self as being of African Berber descent',
       'Identify self as being of Ashkenazi Jewish descent',
       'Identify self as being of Basque descent', 'Birth Date',
       'Sex of participant at birth', 'Female of child bearing potential',
       'How do you live your life day to day?', 'Gay/Lesbian',
       'Straight/Heterosexual', 'Bisexual', 'Pansexual', 'Asexual', 'Other',
       'Handedness', 'Identify ethnicity as being Hispanic or Latino',
       'Identify self as Asian', 'Identify self as Black/African American',
       'Identify self as Hawaiian/Other Pacific Islander',
       'Identify self as American Indian/Alaska Native', 'Race not specified',
       'Identify self as White', 'Race unknown or not reported',
       'Date of original data entry', 'Date of most recent update to record'],
      dtype='object')

In [8]:
relevant_demo_df=['PATNO', 'Birth Date','Sex of participant at birth',
               'Identify self as Asian', 'Identify self as Black/African American',
               'Identify self as Hawaiian/Other Pacific Islander',
               'Identify self as American Indian/Alaska Native',
               'Identify self as White']

demo_df=demo_df[relevant_demo_df]
print(f'Relevant demographics dataframe shape: {demo_df.shape}')
print(f'NaN values in relevant demographics dataframe:\n{demo_df.isna().sum()}')

Relevant demographics dataframe shape: (1056, 8)
NaN values in relevant demographics dataframe:
PATNO                                               0
Birth Date                                          0
Sex of participant at birth                         0
Identify self as Asian                              0
Identify self as Black/African American             0
Identify self as Hawaiian/Other Pacific Islander    0
Identify self as American Indian/Alaska Native      0
Identify self as White                              0
dtype: int64


In [9]:
for col in demo_df.columns[3:]:
      demo_df[col] = demo_df[col].map({'Checked': 1, 'Unchecked': 0})# Convert 'Checked'/'Unchecked' to 1/0

demo_df['Sex of participant at birth']=demo_df['Sex of participant at birth'].map({'Male':1,'Female':0})
demo_df

,PATNO,Birth Date,Sex of participant at birth,Identify self as Asian,Identify self as Black/African American,Identify self as Hawaiian/Other Pacific Islander,Identify self as American Indian/Alaska Native,Identify self as White
1,3001,01/1946,1,0,0,0,0,1
3,3003,07/1954,0,0,0,0,0,1
10,3010,06/1964,1,0,0,0,0,1
18,3018,09/1951,0,0,0,0,0,1
21,3021,04/1948,0,0,0,0,0,1
...,...,...,...,...,...,...,...,...
7405,446733,12/1953,0,0,0,0,0,1
7416,452611,01/1953,0,0,0,0,0,1
7425,466157,01/1997,0,0,0,0,0,1
7433,476236,02/1966,0,0,0,0,0,1


## Socio Economic


In [10]:
socioeco_df = pd.read_csv('/Users/fserracrespi/Desktop/PD_Project_UofL/DATA_COLLECTION/12_09_2025/Subject Characteristics/Subject_Demographics/DATA/Socio-Economics_12Sep2025.csv', dtype=str)
socioeco_df=decoder_DF(socioeco_df, code_rows, code_cols, module='SOCIOECO')
socioeco_df = socioeco_df[socioeco_df['PATNO'].isin(PATNOs)]
socioeco_df=socioeco_df.loc[socioeco_df['Visit ID'].isin(['SC','BL']),:]
socioeco_df['Visit ID']='SC'


/Users/fserracrespi/Desktop/PD_PROJECT_UOFL/utils/Decode.py:81: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df[col].replace('NaN', np.nan, inplace=True)


In [11]:
socioeco_df.columns

Index(['Record ID', 'PATNO', 'Visit ID', 'Page Name', 'Site Aware Date',
       'Number of years of education reported by the participant',
       'Country for education levels', 'Participant's highest education level',
       'Derived education years based on level of education',
       'Date of original data entry', 'Date of most recent update to record'],
      dtype='object')

In [12]:
socioeco_df

,Record ID,PATNO,Visit ID,Page Name,Site Aware Date,Number of years of education reported by the participant,Country for education levels,Participant's highest education level,Derived education years based on level of education,Date of original data entry,Date of most recent update to record
2,274783701,3001,SC,Socio-Economics,02/2011,16,NaN,NaN,NaN,02/2011,2020-06-25 16:04:32.0
6,281141701,3003,SC,Socio-Economics,03/2011,16,NaN,NaN,NaN,03/2011,2020-06-25 16:06:25.0
17,294883501,3010,SC,Socio-Economics,05/2011,16,NaN,NaN,NaN,06/2011,2024-03-21 00:00:00.0
28,336630001,3018,SC,Socio-Economics,02/2012,16,NaN,NaN,NaN,03/2012,2020-06-30 09:25:24.0
31,341540101,3021,SC,Socio-Economics,03/2012,12,NaN,NaN,NaN,04/2012,2020-06-30 09:25:25.0
...,...,...,...,...,...,...,...,...,...,...,...
7369,IA748039,446733,SC,Socio-Economics,05/2025,16,United Kingdom,"Bachelors degree, Bachelor of Nursing",16,05/2025,2025-05-28 00:00:00.0
7380,IA753090,452611,SC,Socio-Economics,03/2025,12,Germany,"Secondary School (Gymnasium, Realschule, Haupt...",12,06/2025,2025-06-04 00:00:00.0
7389,IA744243,466157,SC,Socio-Economics,05/2025,16,NaN,NaN,NaN,05/2025,2025-05-21 00:00:00.0
7397,IA792670,476236,SC,Socio-Economics,07/2025,11,United Kingdom,Secondary School,12,07/2025,2025-07-25 00:00:00.0


In [13]:
col = socioeco_df.columns[5]  # just 1 imputation not a big deal 
socioeco_df[col]=pd.to_numeric(socioeco_df[col], errors='coerce')
socioeco_df[col] = socioeco_df[col].fillna(socioeco_df[col].mean())


In [14]:
socioeco_df.isna().sum()
relevant_socioeco_df=['PATNO',col]
socioeco_df=socioeco_df[relevant_socioeco_df]
print(f'Relevant socio-economic dataframe shape: {socioeco_df.shape}')
print(f'NaN values in relevant socio-economic dataframe:\n{socioeco_df.isna().sum()}')
socioeco_df

Relevant socio-economic dataframe shape: (1056, 2)
NaN values in relevant socio-economic dataframe:
PATNO                                                       0
Number of years of education reported by the participant    0
dtype: int64


,PATNO,Number of years of education reported by the participant
2,3001,16.0
6,3003,16.0
17,3010,16.0
28,3018,16.0
31,3021,12.0
...,...,...
7369,446733,16.0
7380,452611,12.0
7389,466157,16.0
7397,476236,11.0


## Family History

In [15]:
fam_df = pd.read_csv('/Users/fserracrespi/Desktop/PD_Project_UofL/DATA_COLLECTION/12_09_2025/Subject Characteristics/Family_History/DATA/Family_History_12Sep2025.csv', dtype=str)
fam_df=decoder_DF(fam_df, code_rows, code_cols, module='FAMHXPD')
fam_df = fam_df[fam_df['PATNO'].isin(PATNOs)]

irrelevant_cols=['Page Name','Site Aware Date','Biological Mother','Biological Father','Full Siblings','Full Brothers','Full Sisters','Half Siblings',
                'Maternal Half Siblings','Maternal Grandparents','Paternal Grandparents','Maternal Aunts and Uncles',
                'Paternal Aunts and Uncles','Children','Maternal Cousins','Paternal Cousins','Paternal Half Siblings']
fam_df.drop(columns=irrelevant_cols, inplace=True)


fam_df.loc[fam_df["Do you have any family history of Parkinson's Disease or Parkinsonism?"].isna(),"Do you have any family history of Parkinson's Disease or Parkinsonism?"]='No'
fam_df['NA_sum']=fam_df.isna().sum(axis=1)

summary_df =  fam_df[['PATNO', 'Visit ID', "Do you have any family history of Parkinson's Disease or Parkinsonism?",'NA_sum','Date of original data entry']].copy()
summary_df['Date of original data entry'] = pd.to_datetime(summary_df['Date of original data entry'], format="%m/%Y")

# Función de selección por grupo (por PATNO)
def seleccionar_fila(grupo):
    # Si solo hay una fila, devolverla
    if len(grupo) == 1:
        return grupo

    # Si hay alguna fila con "Yes"
    if (grupo['Do you have any family history of Parkinson\'s Disease or Parkinsonism?'] == 'Yes').any():
        grupo_yes = grupo[grupo['Do you have any family history of Parkinson\'s Disease or Parkinsonism?'] == 'Yes']
        # Tomar las de menor NA_sum
        min_na = grupo_yes['NA_sum'].min()
        grupo_min_na = grupo_yes[grupo_yes['NA_sum'] == min_na]
        # Si hay empate, tomar la más antigua
        fila = grupo_min_na.sort_values('Date of original data entry').head(1)
    else:
        # Si todas son "No", tomar la más antigua
        fila = grupo.sort_values('Date of original data entry').head(1)

    return fila

# Aplicar la función por PATNO
summary_df = summary_df.groupby('PATNO', group_keys=False).apply(seleccionar_fila)

# Resultado final
summary_df

fam_df=fam_df.merge(summary_df[['PATNO', 'Visit ID']], on=['PATNO', 'Visit ID'], how='inner')
fam_df.drop(columns=['Date of original data entry','Date of most recent update to record','NA_sum'], inplace=True)
fam_df.isna().sum()



/Users/fserracrespi/Desktop/PD_PROJECT_UOFL/utils/Decode.py:81: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df[col].replace('NaN', np.nan, inplace=True)
/var/folders/vr/_v9wq92941bflv75jsh8kq800000gn/T/ipykernel_47989/3597655166.py:38: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns

Record ID                                                                                                  0
PATNO                                                                                                      0
Visit ID                                                                                                   0
Do you have any family history of Parkinson's Disease or Parkinsonism?                                     0
Biological Mother with PD or Parkinsonism                                                                489
Biological Father with PD or Parkinsonism                                                                488
Full Siblings with PD                                                                                    503
Full Brothers with PD or Parkinsonism                                                                    676
Full Sisters with PD or Parkinsonism                                                                     673
Half Siblings with 

In [16]:
fam_df["Do you have any family history of Parkinson's Disease or Parkinsonism?"].value_counts(dropna=False)

Do you have any family history of Parkinson's Disease or Parkinsonism?
No     650
Yes    406
Name: count, dtype: int64

In [17]:
data_no=fam_df.loc[fam_df["Do you have any family history of Parkinson's Disease or Parkinsonism?"]=='No',:]
ids_no=data_no['PATNO'].unique()

for id,row in fam_df.iterrows():
    if row['PATNO'] in ids_no:
        for col in fam_df.columns:
            if col!='PATNO' and col!='Visit ID':
                fam_df.at[id, col]='No PD Family History'

fam_df.isna().sum()
fam_df.fillna('Unknown', inplace=True)


fam_df.drop(columns=['Visit ID'], inplace=True)

## AGE AT VISIT

In [18]:
Age_df = pd.read_csv('/Users/fserracrespi/Desktop/PD_Project_UofL/DATA_COLLECTION/12_09_2025/Subject Characteristics/Subject_Demographics/DATA/Age_at_visit_12Sep2025.csv', dtype=str)
Age_df.rename(columns={'EVENT_ID':'Visit ID'}, inplace=True)
Age_df=Age_df[Age_df['PATNO'].isin(PATNOs)]
Age_df=Age_df.loc[Age_df['Visit ID'].isin(["BL", "V04", "V06", "V08", "V10", "V12"]),:]
Age_df.isna().sum()

PATNO           0
Visit ID        0
AGE_AT_VISIT    0
dtype: int64

# FINAL DATA

In [19]:
subject_df.head()

,PATNO,Enrollment Cohort,Decoded Value for COHORT,Enrollment Date,Enrollment Status,Date Enrollment Status Occurred,PPMI Clinical Amendment Participant was Screened under,Age at Enrollment,PPMI Clinical Inclusion/Criteria,Participant in PPMI Clinical Early Imaging Sub-Study,...,Pink1 Mutation at Enrollment,Parkin Mutation at Enrollment,Sporadic PD at Enrollment,Normosmic PD at Enrollment,Other Genetic Variant at Enrollment,Hyposmia / Generalized Risk at Enrollment,RBD at Enrollment,LRRK2 Mutation at Enrollment,SNCA Mutation at Enrollment,GBA Mutation at Enrollment
1,3001,Parkinson's Disease,Parkinson's Disease,03/2011,Enrolled,09/2021,NaN,65.1,NaN,No,...,No,No,Yes,NaN,NaN,No,No,No,No,No
3,3003,Parkinson's Disease,Parkinson's Disease,04/2011,Enrolled,01/2022,NaN,56.7,NaN,No,...,No,No,Yes,NaN,NaN,No,No,No,No,No
10,3010,Parkinson's Disease,Parkinson's Disease,06/2011,Enrolled,05/2021,NaN,47.0,NaN,No,...,No,No,Yes,NaN,NaN,No,No,No,No,No
18,3018,Parkinson's Disease,Parkinson's Disease,04/2012,Enrolled,07/2021,NaN,60.6,NaN,No,...,No,No,Yes,NaN,NaN,No,No,No,No,No
21,3021,Parkinson's Disease,Parkinson's Disease,05/2012,Enrolled,06/2021,NaN,64.1,NaN,No,...,No,No,Yes,NaN,NaN,No,No,No,No,No


In [20]:
demo_df.head()

,PATNO,Birth Date,Sex of participant at birth,Identify self as Asian,Identify self as Black/African American,Identify self as Hawaiian/Other Pacific Islander,Identify self as American Indian/Alaska Native,Identify self as White
1,3001,01/1946,1,0,0,0,0,1
3,3003,07/1954,0,0,0,0,0,1
10,3010,06/1964,1,0,0,0,0,1
18,3018,09/1951,0,0,0,0,0,1
21,3021,04/1948,0,0,0,0,0,1


In [21]:
socioeco_df.head()

,PATNO,Number of years of education reported by the participant
2,3001,16.0
6,3003,16.0
17,3010,16.0
28,3018,16.0
31,3021,12.0


In [22]:
Age_df.head()
Age_df['AGE_AT_VISIT']=pd.to_numeric(Age_df['AGE_AT_VISIT'], errors='coerce')
Age_df = Age_df.groupby(['PATNO', 'Visit ID'], as_index=False)['AGE_AT_VISIT'].mean()



In [23]:
SUBJECT_FINAL=pd.merge(relevant_subject_df, demo_df, on='PATNO', how='inner')
SUBJECT_FINAL=pd.merge(SUBJECT_FINAL, socioeco_df, on=['PATNO'], how='inner')
SUBJECT_FINAL=pd.merge(SUBJECT_FINAL, Age_df, on=['PATNO'], how='right')
print(f'Final merged dataframe shape: {SUBJECT_FINAL.shape}')
print(f'PATNOs in final dataframe: {SUBJECT_FINAL["PATNO"].nunique()}')
SUBJECT_FINAL.head()

Final merged dataframe shape: (3999, 18)
PATNOs in final dataframe: 1055


,PATNO,Sporadic PD at Enrollment,RBD at Enrollment,Pink1 Mutation at Enrollment,Parkin Mutation at Enrollment,LRRK2 Mutation at Enrollment,SNCA Mutation at Enrollment,GBA Mutation at Enrollment,Birth Date,Sex of participant at birth,Identify self as Asian,Identify self as Black/African American,Identify self as Hawaiian/Other Pacific Islander,Identify self as American Indian/Alaska Native,Identify self as White,Number of years of education reported by the participant,Visit ID,AGE_AT_VISIT
0,100001,1,0,0,0,0,0,0,05/1953,1,0,0,0,0,1,16.0,BL,67.4
1,100001,1,0,0,0,0,0,0,05/1953,1,0,0,0,0,1,16.0,V04,68.5
2,100001,1,0,0,0,0,0,0,05/1953,1,0,0,0,0,1,16.0,V06,69.5
3,100001,1,0,0,0,0,0,0,05/1953,1,0,0,0,0,1,16.0,V08,70.5
4,100001,1,0,0,0,0,0,0,05/1953,1,0,0,0,0,1,16.0,V10,71.3


In [24]:
# Convertir a fecha
SUBJECT_FINAL['Birth Date'] = pd.to_datetime(
    SUBJECT_FINAL['Birth Date'],
    format='%m/%Y',
    errors='coerce'   # evita que el código falle si hay formatos distintos
)

SUBJECT_FINAL['AGE_AT_VISIT']=pd.to_numeric(SUBJECT_FINAL['AGE_AT_VISIT'], errors='coerce')

# Calcular meses (1 año = 12 meses)
SUBJECT_FINAL['DATE_VISIT'] = SUBJECT_FINAL.apply(lambda x: x['Birth Date'] + pd.DateOffset(months=int(round(x['AGE_AT_VISIT'] * 12))), axis=1)

SUBJECT_FINAL

,PATNO,Sporadic PD at Enrollment,RBD at Enrollment,Pink1 Mutation at Enrollment,Parkin Mutation at Enrollment,LRRK2 Mutation at Enrollment,SNCA Mutation at Enrollment,GBA Mutation at Enrollment,Birth Date,Sex of participant at birth,Identify self as Asian,Identify self as Black/African American,Identify self as Hawaiian/Other Pacific Islander,Identify self as American Indian/Alaska Native,Identify self as White,Number of years of education reported by the participant,Visit ID,AGE_AT_VISIT,DATE_VISIT
0,100001,1,0,0,0,0,0,0,1953-05-01,1,0,0,0,0,1,16.0,BL,67.4,2020-10-01
1,100001,1,0,0,0,0,0,0,1953-05-01,1,0,0,0,0,1,16.0,V04,68.5,2021-11-01
2,100001,1,0,0,0,0,0,0,1953-05-01,1,0,0,0,0,1,16.0,V06,69.5,2022-11-01
3,100001,1,0,0,0,0,0,0,1953-05-01,1,0,0,0,0,1,16.0,V08,70.5,2023-11-01
4,100001,1,0,0,0,0,0,0,1953-05-01,1,0,0,0,0,1,16.0,V10,71.3,2024-09-01
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3994,75562,0,0,0,0,1,0,0,1958-04-01,0,0,0,0,0,1,12.0,V04,61.8,2020-02-01
3995,75562,0,0,0,0,1,0,0,1958-04-01,0,0,0,0,0,1,12.0,V06,62.8,2021-02-01
3996,75562,0,0,0,0,1,0,0,1958-04-01,0,0,0,0,0,1,12.0,V08,64.1,2022-05-01
3997,75562,0,0,0,0,1,0,0,1958-04-01,0,0,0,0,0,1,12.0,V10,64.9,2023-03-01


## implemnetacion de visitas

In [25]:

add_visists = SUBJECT_FINAL.groupby('PATNO')['Visit ID'].apply(list).to_frame().reset_index()
add_age = SUBJECT_FINAL.groupby('PATNO')['AGE_AT_VISIT'].apply(list).to_frame().reset_index()
add_visists = pd.merge(add_visists, add_age, on='PATNO', how='inner')

# Función para identificar visitas faltantes
def adding_visits(row):
    all_visits = ["BL", "V04", "V06", "V08", "V10", "V12"]
    existing_visits = row['Visit ID']
    missing_visits = [visit for visit in all_visits if visit not in existing_visits]
    return missing_visits

add_visists['Missing_Visits'] = add_visists.apply(adding_visits, axis=1)


# calcular edades faltantes con +1 año por visita


all_visits = ["BL", "V04", "V06", "V08", "V10", "V12"]
visit_year_offset = {"BL":0, "V04":1, "V06":2, "V08":3, "V10":4, "V12":5}

def compute_missing_ages(row):
    visit_list = row["Visit ID"]
    age_list = row["AGE_AT_VISIT"]

    # Edad basal siempre de BL
    baseline_age = age_list[visit_list.index("BL")]

    # Calcular edades faltantes
    missing_ages = [
        baseline_age + visit_year_offset[v]
        for v in row["Missing_Visits"]
    ]
    return missing_ages

# Añadir la columna de edades faltantes calculadas
add_visists['Missing_Ages'] = add_visists.apply(compute_missing_ages, axis=1)

# Ver distribución de visitas faltantes
add_visists




,PATNO,Visit ID,AGE_AT_VISIT,Missing_Visits,Missing_Ages
0,100001,"[BL, V04, V06, V08, V10, V12]","[67.4, 68.5, 69.5, 70.5, 71.3, 72.3]",[],[]
1,100002,"[BL, V04, V08, V10]","[58.5, 59.6, 61.6, 62.9]","[V06, V12]","[60.5, 63.5]"
2,100005,"[BL, V04, V06, V08]","[52.8, 53.8, 55.8, 56.1]","[V10, V12]","[56.8, 57.8]"
3,100006,"[BL, V04, V06, V08, V10]","[55.7, 56.7, 57.9, 58.9, 59.8]",[V12],[60.7]
4,100007,"[BL, V04, V06, V08, V10]","[67.2, 68.4, 69.5, 70.4, 71.4]",[V12],[72.2]
...,...,...,...,...,...
1050,75480,"[BL, V04, V06, V08, V10, V12]","[56.3, 57.3, 58.5, 59.3, 60.6, 61.5]",[],[]
1051,75484,"[BL, V04, V06, V08, V10, V12]","[57.6, 58.7, 59.7, 60.7, 61.6, 62.6]",[],[]
1052,75505,"[BL, V04, V06, V08, V10, V12]","[64.0, 65.0, 66.5, 67.0, 68.3, 69.0]",[],[]
1053,75524,"[BL, V04, V06, V08, V10, V12]","[54.7, 55.6, 56.9, 57.8, 58.9, 59.7]",[],[]


In [26]:
all_visits = ["BL", "V04", "V06", "V08", "V10", "V12"]
visit_year_offset = {"BL":0, "V04":1, "V06":2, "V08":3, "V10":4, "V12":5}

new_rows = []

for patno, group in SUBJECT_FINAL.groupby("PATNO"):
    
    existing_visits = group["Visit ID"].tolist()
    
    # Obtener la fila BL completa
    bl_row = group[group["Visit ID"] == "BL"].iloc[0].copy()
    baseline_age = bl_row["AGE_AT_VISIT"]
    date_visit_bl = bl_row["DATE_VISIT"]
    
    missing = [v for v in all_visits if v not in existing_visits]
    
    for v in missing:
        new_row = bl_row.copy()  # copiar todas las columnas del BL
        new_row["Visit ID"] = v
        new_row["AGE_AT_VISIT"] = baseline_age + visit_year_offset[v]
        new_row["DATE_VISIT"] = date_visit_bl + pd.DateOffset(years=visit_year_offset[v])
        new_rows.append(new_row)


missing_df = pd.DataFrame(new_rows)
SUBJECT_FINAL = pd.concat([SUBJECT_FINAL, missing_df], ignore_index=True)
SUBJECT_FINAL.sort_values(by=['PATNO', 'Visit ID'], inplace=True)

In [27]:
SUBJECT_FINAL.loc[SUBJECT_FINAL['PATNO']=='100002',:]

,PATNO,Sporadic PD at Enrollment,RBD at Enrollment,Pink1 Mutation at Enrollment,Parkin Mutation at Enrollment,LRRK2 Mutation at Enrollment,SNCA Mutation at Enrollment,GBA Mutation at Enrollment,Birth Date,Sex of participant at birth,Identify self as Asian,Identify self as Black/African American,Identify self as Hawaiian/Other Pacific Islander,Identify self as American Indian/Alaska Native,Identify self as White,Number of years of education reported by the participant,Visit ID,AGE_AT_VISIT,DATE_VISIT
6,100002,1,0,0,0,0,0,0,1962-04-01,1,0,0,0,0,1,18.0,BL,58.5,2020-10-01
7,100002,1,0,0,0,0,0,0,1962-04-01,1,0,0,0,0,1,18.0,V04,59.6,2021-11-01
3999,100002,1,0,0,0,0,0,0,1962-04-01,1,0,0,0,0,1,18.0,V06,60.5,2022-10-01
8,100002,1,0,0,0,0,0,0,1962-04-01,1,0,0,0,0,1,18.0,V08,61.6,2023-11-01
9,100002,1,0,0,0,0,0,0,1962-04-01,1,0,0,0,0,1,18.0,V10,62.9,2025-03-01
4000,100002,1,0,0,0,0,0,0,1962-04-01,1,0,0,0,0,1,18.0,V12,63.5,2025-10-01


In [28]:
SUBJECT_FINAL.shape

(6330, 19)

In [29]:
SUBJECT_FINAL.to_csv('/Users/fserracrespi/Desktop/PD_Project_UofL/PD_DATA/PD_CSV_CLEAN/FINAL DATA TEST CLEAN/FINAL_SUBJECT_CHARACTERISTICS_12SEP2025.csv', index=False)

In [30]:
SUBJECT_FINAL.groupby('PATNO')['Visit ID'].value_counts()

PATNO   Visit ID
100001  BL          1
        V04         1
        V06         1
        V08         1
        V10         1
                   ..
75562   V04         1
        V06         1
        V08         1
        V10         1
        V12         1
Name: count, Length: 6330, dtype: int64

In [31]:
SUBJECT_FINAL['PATNO'].nunique()

1055

# ANALYSIS DE PROGRESSION

In [32]:
progression_summary = check_progression(SUBJECT_FINAL)
progression_summary.reset_index('PATNO', inplace=True)

path='/Users/fserracrespi/Desktop/PD_Project_UofL/PD_DATA/PROGRESSION_STUDIES'
for col in progression_summary.columns[1:]:
    path_col=path+'/'+col+'_12SEP2025.csv'
    progression_summary[['PATNO', col]].to_csv(path_col, index=False)

In [33]:
progression_summary_multi=check_progression_multi(SUBJECT_FINAL, name_prefix="SUBJECT")
progression_summary_multi.reset_index('PATNO', inplace=True)

path='/Users/fserracrespi/Desktop/PD_Project_UofL/PD_DATA/PROGRESSION_STUDIES'

for col in progression_summary_multi.columns[1:]:
    path_col=path+'/MULTIPLE_'+col+'12SEP2025.csv'
    progression_summary_multi[['PATNO', col]].to_csv(path_col, index=False)


In [34]:
check_visits_summary=check_visits(SUBJECT_FINAL, name_prefix="SUBJECT")

path='/Users/fserracrespi/Desktop/PD_Project_UofL/PD_DATA/PROGRESSION_STUDIES'
for col in check_visits_summary.columns[1:]:
    path_col=path+'/VISITS_'+col+'_12SEP2025.csv'
    check_visits_summary[['PATNO', col]].to_csv(path_col, index=False)


In [35]:
cols_asignacion2(SUBJECT_FINAL,df_secundario='Subject_Demographics_SocioEconomics_FHistory_AgeVisit',df_main='SUBJECT')


,df_main,df_secundario,df_secundario_col
0,SUBJECT,Subject_Demographics_SocioEconomics_FHistory_A...,PATNO
1,SUBJECT,Subject_Demographics_SocioEconomics_FHistory_A...,Sporadic PD at Enrollment
2,SUBJECT,Subject_Demographics_SocioEconomics_FHistory_A...,RBD at Enrollment
3,SUBJECT,Subject_Demographics_SocioEconomics_FHistory_A...,Pink1 Mutation at Enrollment
4,SUBJECT,Subject_Demographics_SocioEconomics_FHistory_A...,Parkin Mutation at Enrollment
5,SUBJECT,Subject_Demographics_SocioEconomics_FHistory_A...,LRRK2 Mutation at Enrollment
6,SUBJECT,Subject_Demographics_SocioEconomics_FHistory_A...,SNCA Mutation at Enrollment
7,SUBJECT,Subject_Demographics_SocioEconomics_FHistory_A...,GBA Mutation at Enrollment
8,SUBJECT,Subject_Demographics_SocioEconomics_FHistory_A...,Birth Date
9,SUBJECT,Subject_Demographics_SocioEconomics_FHistory_A...,Sex of participant at birth
